# Navegador Prompt Testing Workbench

Interactive test workflow for all LLM prompts in the Navegador project.

## What this notebook covers:
1. **Intent Classification** - Route user queries to the right handler
2. **Project Description** - Answer general questions about the survey project
3. **Variable Grading** - Grade relevance of survey variables to user queries
4. **Cross-Analysis Pattern Extraction** - Find similar/different patterns in survey data
5. **Expert Summary** - Generate domain-expert analysis of patterns
6. **Transversal Analysis** - Synthesize cross-topic findings into final report
7. **Meta-Prompting A/B Testing** - Compare prompt versions (V1 vs V2 vs V3)

Each test uses **actual survey data** from the `structured_summary_checkpoint.json`.

In [24]:
# === SETUP ===
import sys, os, json, time
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

# Project root
PROJECT_ROOT = Path('/workspaces/navegador')
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

# Verify API keys
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not set'
print('OpenAI API key: OK')
print(f'Anthropic API key: {"OK" if os.getenv("ANTHROPIC_API_KEY") else "MISSING (optional)"}')

OpenAI API key: OK
Anthropic API key: OK


In [25]:
# === LOAD REAL SURVEY DATA ===
# This file contains actual preprocessed survey summaries and implications
with open(PROJECT_ROOT / 'data/results/structured_summary_checkpoint.json', 'r') as f:
    checkpoint_data = json.load(f)

print(f'Loaded {len(checkpoint_data)} survey variables from checkpoint')
print(f'Sample variable IDs: {list(checkpoint_data.keys())[:5]}')

# Show a sample entry
sample_key = list(checkpoint_data.keys())[0]
sample_val = checkpoint_data[sample_key]
display(Markdown(f'### Sample: `{sample_key}`'))
display(Markdown(f'**Summary:** {sample_val["summary"][:200]}...'))
display(Markdown(f'**Implication:** {sample_val["implication"][:200]}...'))

Loaded 4423 survey variables from checkpoint
Sample variable IDs: ['p1_1a_1|IDE', 'p2_1a_1|IDE', 'p3_1|IDE', 'p3_2|IDE', 'p3_3|IDE']


### Sample: `p1_1a_1|IDE`

**Summary:** The table presents various associations that people have with the word 'México', showcasing a wide range of responses. The most prominent association is 'Orgullo', which reflects a strong sense of nat...

**Implication:** This table is crucial for experts in 'identidad y valores' as it provides insights into the perceptions and feelings of individuals regarding their national identity. Understanding these associations ...

In [26]:
# === HELPER: LLM Call Function ===
from openai import OpenAI
import re

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def call_llm(prompt, model='gpt-4.1-mini-2025-04-14', temperature=0.5, system_prompt=None):
    """Direct LLM call with timing and token tracking."""
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    
    start = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    elapsed = time.time() - start
    
    content = response.choices[0].message.content
    usage = response.usage
    
    print(f'  Model: {model}')
    print(f'  Tokens: {usage.prompt_tokens} prompt + {usage.completion_tokens} completion = {usage.total_tokens} total')
    print(f'  Latency: {elapsed:.2f}s')
    
    return content, {'tokens': usage.total_tokens, 'latency': elapsed, 'model': model}

def clean_json(content):
    """Clean LLM output to valid JSON."""
    content = content.strip()
    content = re.sub(r'^```json', '', content, flags=re.IGNORECASE).strip()
    content = re.sub(r'^```', '', content, flags=re.IGNORECASE).strip()
    content = re.sub(r'```$', '', content, flags=re.IGNORECASE).strip()
    idx = content.find('{')
    if idx > 0:
        content = content[idx:]
    idx2 = content.rfind('}')
    if idx2 > 0:
        content = content[:idx2+1]
    return content

print('LLM helper ready.')

LLM helper ready.


---
## 1. Intent Classification Prompt

**Purpose:** Classify user messages into one of 8 intent types to route the workflow.

**Source:** `intent_classifier.py:41-76`

In [27]:
# === TEST 1: INTENT CLASSIFICATION ===

intent_dict = {
    'answer_general_questions': 'If the user asks questions about the project, the datasets, the methodology or the team.',
    'continue_conversation': 'If the user asks what you can do.',
    'query_variable_database': 'If the user asks a question that should be answered by querying survey variables.',
    'review_variable_selection': 'If the user asks to review the selection of variables.',
    'select_analysis_type': 'If the user asks for simple or complex analysis.',
    'confirm_and_run': 'If the user asks to produce an analysis or report.',
    'reset_conversation': 'If the user asks to start over.',
    'end_conversation': 'If the user asks to end the conversation.'
}

intent_str = '\n'.join([f'*{act}: {cond}' for act, cond in intent_dict.items()])

def test_intent_classification(user_message):
    prompt = f"""
    You are an intent classifier for a dataset analysis assistant.
    Classify the user's intent into exactly one of the available intents.

    Available intents:
    {intent_str}

    User message: "{user_message}"

    CRITICAL DISTINCTION:
    - Questions ABOUT the data/variables themselves ("What do people think about X?") -> query_variable_database
    - Questions ABOUT what data/topics are AVAILABLE ("What topics do you have?") -> answer_general_questions

    Respond with JSON: {{"intent": "...", "confidence": 0.0}}
    """
    
    display(Markdown(f'**Input:** "{user_message}"'))
    content, meta = call_llm(prompt, temperature=0.0)
    cleaned = clean_json(content)
    result = json.loads(cleaned)
    display(Markdown(f'**Result:** `{result["intent"]}` (confidence: {result.get("confidence", "N/A")})'))
    return result

# Test cases
test_queries = [
    'What do Mexicans think about education?',           # -> query_variable_database
    'What topics are available?',                         # -> answer_general_questions
    'How do people feel about corruption?',               # -> query_variable_database
    'What is this project about?',                        # -> answer_general_questions
    'Run the analysis',                                   # -> confirm_and_run
    'I want a detailed analysis',                         # -> select_analysis_type
]

print('=== INTENT CLASSIFICATION TEST ===')
for q in test_queries:
    print()
    test_intent_classification(q)
    print('-' * 60)

=== INTENT CLASSIFICATION TEST ===



**Input:** "What do Mexicans think about education?"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 254 prompt + 16 completion = 270 total
  Latency: 0.74s


**Result:** `query_variable_database` (confidence: 1.0)

------------------------------------------------------------



...

------------------------------------------------------------



**Input:** "I want a detailed analysis"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 252 prompt + 16 completion = 268 total
  Latency: 0.73s


**Result:** `select_analysis_type` (confidence: 1.0)

------------------------------------------------------------


In [28]:
# === TRY YOUR OWN INTENT QUERY ===
# Change this message and re-run the cell
YOUR_MESSAGE = "What do Mexicans think about democracy?"

test_intent_classification(YOUR_MESSAGE)

**Input:** "What do Mexicans think about democracy?"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 254 prompt + 16 completion = 270 total
  Latency: 0.49s


**Result:** `query_variable_database` (confidence: 1.0)

{'intent': 'query_variable_database', 'confidence': 1.0}

---
## 2. Project Description Prompt

**Purpose:** Answer general questions about the "Los Mexicanos" survey project.

**Source:** `dataset_knowledge.py:140-152`

In [29]:
# === TEST 2: PROJECT DESCRIPTION ===
# TODO: populate topic list with all 26 surveys
# TODO: questions about topics return a reommendation to visit the website, 
#.      it should redirect to the analysis procedures, so make sure it pipes to variable selection 
#.      or straight to querring question datasets and writing a report
# TODO: create a skills.md or an mcp that instructs the orchestrator what it can do


# 
# Real project metadata
topics_list = [
    'Identidad Y Valores', 'Medio Ambiente', 'Ciencia Y Tecnología',
    'Cultura', 'Derechos Humanos', 'Discriminación',
    'Economía', 'Educación', 'Familia', 'Justicia',
    'Migración', 'Pobreza', 'Política', 'Religión',
    'Salud', 'Seguridad', 'Sociedad'
]
topics_str = ', '.join(topics_list)

project_description = f"""
General information about the project:
-Name: "Los mexicanos vistos por sí mismos"
-Description: This project is a group of {len(topics_list)} public opinion surveys conducted in Mexico between 2014 and 2015.
-Topics: {topics_str}.
-Objective: To understand the Mexican population's opinions on various topics.
-Team: Mtra. Julia Flores coordinated the team at the "Unidad de Investigación en Opinión Pública".
-Repository: http://www.losmexicanos.unam.mx/
-Sponsor: Universidad Nacional Autónoma de México (UNAM)
-Samples: All samples have size 1000, margin of error 3%, confidence 95%.
"""

def test_project_describer(user_query):
    prompt = f"""
    You are a helpful assistant answering general questions about a survey project.
    Read the project description and reply with relevant information.
    IMPORTANT: If the user asks about specific variables, redirect them to QUERY VARIABLE_DATABASE.

    Project description:
    {project_description}

    User query: "{user_query}"

    Respond with JSON: {{"answer": "...", "redirect_needed": false, "suggested_action": ""}}
    """
    
    display(Markdown(f'**Query:** "{user_query}"'))
    content, meta = call_llm(prompt, temperature=0.3)
    cleaned = clean_json(content)
    result = json.loads(cleaned)
    display(Markdown(f'**Answer:** {result["answer"]}'))
    if result.get('redirect_needed'):
        display(Markdown(f'*Redirect suggested: {result["suggested_action"]}*'))
    return result

print('=== PROJECT DESCRIPTION TEST ===')
test_project_describer('What topics are covered in this survey project?')
print()
test_project_describer('Who funded this project?')
print()
test_project_describer('What do Mexicans think about healthcare?')  # Should redirect

=== PROJECT DESCRIPTION TEST ===


**Query:** "What topics are covered in this survey project?"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 269 prompt + 73 completion = 342 total
  Latency: 1.00s


**Answer:** The survey project covers the following topics: Identidad Y Valores, Medio Ambiente, Ciencia Y Tecnología, Cultura, Derechos Humanos, Discriminación, Economía, Educación, Familia, Justicia, Migración, Pobreza, Política, Religión, Salud, Seguridad, and Sociedad.

**Query:** "Who funded this project?"

...

**Answer:** The project was funded by the Universidad Nacional Autónoma de México (UNAM).

**Query:** "What do Mexicans think about healthcare?"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 268 prompt + 105 completion = 373 total
  Latency: 1.77s


**Answer:** The project 'Los mexicanos vistos por sí mismos' includes surveys that cover the topic of Salud (healthcare). These surveys provide insights into the Mexican population's opinions and perceptions about healthcare services, access, quality, and related issues during 2014-2015. For detailed analysis or specific questions about healthcare, the survey data can be consulted.

{'answer': "The project 'Los mexicanos vistos por sí mismos' includes surveys that cover the topic of Salud (healthcare). These surveys provide insights into the Mexican population's opinions and perceptions about healthcare services, access, quality, and related issues during 2014-2015. For detailed analysis or specific questions about healthcare, the survey data can be consulted.",
 'redirect_needed': False,
 'suggested_action': 'You may explore the survey results on the Salud topic in the project repository for more detailed information.'}

In [30]:
# === TRY YOUR OWN PROJECT QUESTION ===
YOUR_QUESTION = "quien coordino el estudio?"#"How many surveys were conducted and what was the sample size?"

test_project_describer(YOUR_QUESTION)

**Query:** "quien coordino el estudio?"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 267 prompt + 39 completion = 306 total
  Latency: 0.91s


**Answer:** El estudio fue coordinado por la Mtra. Julia Flores en la Unidad de Investigación en Opinión Pública.

{'answer': 'El estudio fue coordinado por la Mtra. Julia Flores en la Unidad de Investigación en Opinión Pública.',
 'redirect_needed': False,
 'suggested_action': ''}

---
## 3. Variable Relevance Grading Prompt

**Purpose:** Grade how relevant a survey variable is to a user's query (0-3 scale).

**Source:** `variable_selector.py:37-83`

In [31]:
# === TEST 3: VARIABLE RELEVANCE GRADING ===

def test_variable_grading(user_query, variable_id, survey_info):
    """Grade a single variable's relevance to a query."""
    prompt = f"""
You are an expert in survey research, fluent in English and Spanish. Reply in English only.
Grade the relevance of the SURVEY INFORMATION to the user QUERY.

THE SURVEY INFORMATION has 3 parts:
- QUESTION: the question asked in the survey
- SUMMARY: a summary of the survey results
- IMPLICATIONS: expert analysis of the results

GRADE (0-3):
- 0: Completely unrelated
- 1: Some connection but mostly unrelated
- 2: Moderately relevant - moderately related, but adding important insights
- 3: Highly relevant - directly addresses the query

Return strict JSON: {{"GRADE": number, "EXPLANATION": "text"}}

QUERY: {user_query}
SURVEY_INFORMATION: {survey_info}
"""
    
    content, meta = call_llm(prompt, temperature=0.0)
    cleaned = clean_json(content)
    result = json.loads(cleaned)
    
    grade = result['GRADE']
    grade_emoji = ['', '🟡', '🟠', '🟢'][int(grade)] if grade <= 3 else ''
    display(Markdown(f'  **{variable_id}**: Grade {grade}/3 {grade_emoji} — {result["EXPLANATION"]}'))
    return result

# Test with real data
query = "What do Mexicans think about national identity and pride?"
display(Markdown(f'### Query: "{query}"'))
print()

# Pick 5 variables (mix of relevant and unrelated)
test_vars = ['p5_1|IDE', 'p6|IDE', 'p3_5|IDE', 'p10_1|IDE', 'p9|IDE']
for var_id in test_vars:
    if var_id in checkpoint_data:
        data = checkpoint_data[var_id]
        info = f"SUMMARY: {data['summary']} IMPLICATIONS: {data['implication']}"
        test_variable_grading(query, var_id, info)
        print()

### Query: "What do Mexicans think about national identity and pride?"


  Model: gpt-4.1-mini-2025-04-14
  Tokens: 365 prompt + 68 completion = 433 total
  Latency: 1.01s


  **p5_1|IDE**: Grade 3/3 🟢 — The survey information directly addresses the query by presenting data on Mexicans' emotional sentiments toward Mexico, highlighting pride as the predominant feeling. The implications further explain how these sentiments relate to national identity and pride, making the information highly relevant to understanding what Mexicans think about these topics.


  Model: gpt-4.1-mini-2025-04-14
  Tokens: 357 prompt + 69 completion = 426 total
  Latency: 1.43s


  **p6|IDE**: Grade 3/3 🟢 — The survey information directly addresses the query by providing detailed data on how proud Mexicans feel about their national identity, including percentages of varying levels of pride. The implications further analyze the significance of these feelings for understanding national identity and pride, making the information highly relevant to the user's question.

...

  **p3_5|IDE**: Grade 3/3 🟢 — The survey information directly addresses the query by providing data on how Mexicans feel emotionally connected to their country, which is a core aspect of national identity and pride. The summary quantifies the levels of attachment, and the implications explain the relevance of these feelings to identity and pride, making it highly relevant to understanding Mexican perspectives on national identity and pride.


  Model: gpt-4.1-mini-2025-04-14
  Tokens: 345 prompt + 79 completion = 424 total
  Latency: 1.40s


  **p10_1|IDE**: Grade 1/3 🟡 — The survey information relates to Mexicans' experiences of cultural diminishment in the workplace, which touches on aspects of national identity and pride. However, it does not directly address broader attitudes or beliefs about national identity and pride itself, focusing instead on workplace cultural sensitivity. Therefore, it has some connection but is mostly unrelated to the query.


  Model: gpt-4.1-mini-2025-04-14
  Tokens: 308 prompt + 90 completion = 398 total
  Latency: 1.54s


  **p9|IDE**: Grade 2/3 🟠 — The survey information provides insights into Mexican attitudes toward ethnic and cultural groups, highlighting a strong preference for respecting diverse cultures and customs. This relates moderately to the query about national identity and pride by indicating a collective value placed on multiculturalism and cultural preservation, which are key components of national identity. However, it does not directly address broader aspects of national pride or identity beyond cultural respect, limiting its full relevance.

In [32]:
# === TRY YOUR OWN GRADING QUERY ===
YOUR_QUERY = "How do people feel about corruption in Mexico?"
# Pick any variable from checkpoint_data.keys()
VAR_ID = 'p4|IDE'  # This is about national identity - probably low relevance to corruption

display(Markdown(f'### Query: "{YOUR_QUERY}"'))
data = checkpoint_data[VAR_ID]
info = f"SUMMARY: {data['summary']} IMPLICATIONS: {data['implication']}"
test_variable_grading(YOUR_QUERY, VAR_ID, info)

### Query: "How do people feel about corruption in Mexico?"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 314 prompt + 59 completion = 373 total
  Latency: 1.02s


  **p4|IDE**: Grade 0/3  — The survey information focuses on opinions about national identity and cultural values in Mexico, with no mention or analysis of corruption or public sentiment towards corruption. Therefore, it is completely unrelated to the query about how people feel about corruption in Mexico.

{'GRADE': 0,
 'EXPLANATION': 'The survey information focuses on opinions about national identity and cultural values in Mexico, with no mention or analysis of corruption or public sentiment towards corruption. Therefore, it is completely unrelated to the query about how people feel about corruption in Mexico.'}

---
## 4. Cross-Analysis Pattern Extraction (Main Analysis Prompt)

**Purpose:** Find similar and different patterns across survey results. This is the core analysis prompt.

**Source:** `detailed_analysis.py:77-140` + `meta_prompting.py` (V1, V2, V3)

In [33]:
# === BUILD REAL SURVEY RESULTS STRING ===
# Prepare actual data for cross-analysis testing

# Select identity-related variables for a focused test
identity_vars = {k: v for k, v in checkpoint_data.items() 
                 if '|IDE' in k and k in list(checkpoint_data.keys())[:15]}

# Build the results string in the format the prompt expects
result_entries = []
for i, (var_id, data) in enumerate(identity_vars.items(), 1):
    result_entries.append(f'* QUESTION_ID: {var_id}')
    result_entries.append(f'  SUMMARY: {data["summary"]}')

survey_results_str = '\n'.join(result_entries)

print(f'Prepared {len(identity_vars)} survey variables for analysis')
print(f'Variables: {list(identity_vars.keys())}')
print(f'Results string length: {len(survey_results_str)} chars')

Prepared 15 survey variables for analysis
Variables: ['p1_1a_1|IDE', 'p2_1a_1|IDE', 'p3_1|IDE', 'p3_2|IDE', 'p3_3|IDE', 'p3_4|IDE', 'p3_5|IDE', 'p3_6|IDE', 'p4|IDE', 'p5_1|IDE', 'p5_1a|IDE', 'p6|IDE', 'p7|IDE', 'p8|IDE', 'p9|IDE']
Results string length: 7306 chars


In [34]:
# === TEST 4a: CROSS-ANALYSIS V1 (Original/Baseline) ===

from meta_prompting import PromptTemplates
# TODO: explain what meta_prompting and PrompTemplated do, 
#.      and how different prompt templated are built
query = "What do Mexicans think about their national identity?"
n_topics = 3

format_instr = 'Return strict JSON only. Keys: SIMILAR_PATTERN_1..N, DIFFERENT_PATTERN_1..N. Each has TITLE_SUMMARY, VARIABLE_STRING, DESCRIPTION.'

prompt_v1 = PromptTemplates.CROSS_ANALYSIS_V1.format(
    n_topics=n_topics,
    user_query=query,
    results=survey_results_str,
    format_instructions=format_instr
)

display(Markdown('### Cross-Analysis V1 (Original Baseline)'))
display(Markdown(f'**Query:** {query}'))
print(f'Prompt length: {len(prompt_v1)} chars (~{len(prompt_v1.split())} words)')
print()

content_v1, meta_v1 = call_llm(prompt_v1, temperature=0.5)
cleaned_v1 = clean_json(content_v1)
result_v1 = json.loads(cleaned_v1)

print()
for key, val in result_v1.items():
    if 'PATTERN' in key:
        display(Markdown(f'**{key}**: {val["TITLE_SUMMARY"]}'))
        display(Markdown(f'  Variables: `{val["VARIABLE_STRING"]}`'))
        display(Markdown(f'  {val["DESCRIPTION"][:300]}'))
        print()

### Cross-Analysis V1 (Original Baseline)

**Query:** What do Mexicans think about their national identity?

Prompt length: 8621 chars (~1268 words)

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 2116 prompt + 878 completion = 2994 total
  Latency: 9.66s



**SIMILAR_PATTERN_1**: Strong National Pride Among Mexicans

  Variables: `p1_1a_1,p5_1,p5_1a,p6`

...

  Variables: `p3_5,p3_6`

  Respondents feel a much stronger emotional connection to Mexico (81.25% 'Mucho' or 'Algo') (p3_5) compared to their connection to the world, where only 62.08% feel 'Mucho' or 'Algo' connected, and a significant 31.92% feel little or no connection (p3_6). This indicates a more inward-focused national

**DIFFERENT_PATTERN_3**: Divergence in Identity Between National and Local (Yucateco) Identity

  Variables: `p7`

  A notable portion of respondents identify exclusively as Mexican (42.00%), but a significant minority identifies more strongly with their local identity, such as Yucateco (15.00% 'More Yucateco than Mexican' plus 4.42% who identify exclusively as Yucateco) (p7). This highlights a tension or distinct

In [35]:
# === TEST 4b: CROSS-ANALYSIS V2 (Optimized/Concise) ===

prompt_v2 = PromptTemplates.CROSS_ANALYSIS_V2.format(
    n_topics=n_topics,
    user_query=query,
    results=survey_results_str,
    format_instructions=format_instr
)

display(Markdown('### Cross-Analysis V2 (Optimized Concise)'))
print(f'Prompt length: {len(prompt_v2)} chars (~{len(prompt_v2.split())} words)')
print(f'Reduction from V1: {(1 - len(prompt_v2)/len(prompt_v1))*100:.1f}%')
print()

content_v2, meta_v2 = call_llm(prompt_v2, temperature=0.5)
cleaned_v2 = clean_json(content_v2)
result_v2 = json.loads(cleaned_v2)

print()
for key, val in result_v2.items():
    if 'PATTERN' in key:
        display(Markdown(f'**{key}**: {val["TITLE_SUMMARY"]}'))
        display(Markdown(f'  Variables: `{val["VARIABLE_STRING"]}`'))
        display(Markdown(f'  {val["DESCRIPTION"][:300]}'))
        print()

### Cross-Analysis V2 (Optimized Concise)

Prompt length: 8105 chars (~1173 words)
Reduction from V1: 6.0%

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 2005 prompt + 944 completion = 2949 total
  Latency: 9.65s



**SIMILAR_PATTERN_1**: Strong Emotional Attachment to Mexico and Subnational Identities

  Variables: `p3_1|IDE,p3_2|IDE,p3_3|IDE,p3_4|IDE,p3_5|IDE`

  Respondents show strong feelings of connection at multiple geographic levels: 37.92% feel 'Mucho' attached to their neighborhood (p3_1|IDE), 39.75% to their locality (p3_2|IDE), 41.75% to their state (p3_3|IDE), 39.58% to their region (p3_4|IDE), and 48.17% feel 'Mucho' attached to Mexico as a whole

...

  Variables: `p3_5|IDE,p3_6|IDE`

  While 48.17% feel 'Mucho' connected to Mexico (p3_5|IDE), only 27.33% feel 'Mucho' connected to the world (p3_6|IDE). Moreover, 31.92% feel little or no connection to the world ('Poco' 22.00%, 'Nada' 9.92%) (p3_6|IDE), indicating a marked difference in emotional attachment between national/local and

**DIFFERENT_PATTERN_3**: Diverse Self-Identification Between National and Regional Identity

  Variables: `p7|IDE`

  A plurality (42.00%) identify as 'Sólo mexicano', while a notable 15.00% identify as 'Más (yucateco) que mexicano' and an additional 4.42% identify exclusively as Yucateco (p7|IDE). This shows a significant divergence in identity emphasis between national and regional/local affiliations.

In [36]:
# === TEST 4c: CROSS-ANALYSIS V3 (Chain-of-Thought) ===

prompt_v3 = PromptTemplates.CROSS_ANALYSIS_V3.format(
    n_topics=n_topics,
    user_query=query,
    results=survey_results_str,
    format_instructions=format_instr
)

display(Markdown('### Cross-Analysis V3 (Chain-of-Thought)'))
print(f'Prompt length: {len(prompt_v3)} chars (~{len(prompt_v3.split())} words)')
print(f'Reduction from V1: {(1 - len(prompt_v3)/len(prompt_v1))*100:.1f}%')
print()

content_v3, meta_v3 = call_llm(prompt_v3, temperature=0.5)
cleaned_v3 = clean_json(content_v3)
result_v3 = json.loads(cleaned_v3)

print()
for key, val in result_v3.items():
    if 'PATTERN' in key:
        display(Markdown(f'**{key}**: {val["TITLE_SUMMARY"]}'))
        display(Markdown(f'  Variables: `{val["VARIABLE_STRING"]}`'))
        display(Markdown(f'  {val["DESCRIPTION"][:300]}'))
        print()

### Cross-Analysis V3 (Chain-of-Thought)

Prompt length: 8231 chars (~1194 words)
Reduction from V1: 4.5%

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 2021 prompt + 1082 completion = 3103 total
  Latency: 13.88s



**SIMILAR_PATTERN_1**: Strong sense of pride and positive emotional connection to Mexico

  Variables: `p1_1a_1|IDE,p5_1a|IDE,p6|IDE`

  Respondents consistently express pride related to Mexico and their national identity. In p1_1a_1|IDE, 'Orgullo' is the most prominent association with Mexico at 25.25%. This is reinforced in p5_1a|IDE where 'Orgullo' (Pride) is mentioned by 38.33% of respondents as an emotion associated with Mexico.

...

  Variables: `p3_2|IDE,p3_5|IDE,p3_6|IDE`

  Respondents feel more connected to their locality and country than to the world at large. In p3_2|IDE, 79.83% feel 'Mucho' or 'Algo' connected to their locality, and in p3_5|IDE, 81.25% feel 'Mucho' or 'Algo' connected to Mexico. Conversely, p3_6|IDE shows only 27.33% feel 'Mucho' connected to the w

**DIFFERENT_PATTERN_3**: High value placed on preserving local traditions contrasts with mixed feelings about national emotional sentiments

  Variables: `p8|IDE,p5_1a|IDE`

  There is strong emphasis on preserving traditions with 73% valuing it 'Mucho' in p8|IDE, demonstrating cultural continuity at local levels. However, emotional sentiments towards Mexico in p5_1a|IDE are mixed: although pride is high at 38.33%, concern (17.00%) and anger (11.25%) also represent substa

In [37]:
# === COMPARE ALL 3 VERSIONS SIDE-BY-SIDE ===

from prompt_integration import PromptQualityAssessor
# TODO: explain what PromptQualityAssessor does and how it works
# TODO: explain how V2 is different from V1

assessor = PromptQualityAssessor()

versions = {
    'V1 (Baseline)':  {'content': content_v1, 'result': result_v1, 'meta': meta_v1, 'prompt_len': len(prompt_v1)},
    'V2 (Optimized)': {'content': content_v2, 'result': result_v2, 'meta': meta_v2, 'prompt_len': len(prompt_v2)},
    'V3 (CoT)':       {'content': content_v3, 'result': result_v3, 'meta': meta_v3, 'prompt_len': len(prompt_v3)},
}

display(Markdown('## Cross-Analysis Version Comparison'))
display(Markdown(f'**Query:** "{query}" | **Patterns requested:** {n_topics} similar + {n_topics} different'))
print()

comparison_rows = []
for name, data in versions.items():
    quality = assessor.assess_cross_analysis_quality(
        data['content'], data['result'], n_topics * 2
    )
    patterns_found = len([k for k in data['result'] if 'PATTERN' in k])
    
    # Count question ID citations
    citations = data['content'].count('|')
    
    comparison_rows.append({
        'name': name,
        'prompt_words': len(data['content'].split()),
        'tokens': data['meta']['tokens'],
        'latency': data['meta']['latency'],
        'patterns': patterns_found,
        'quality': quality,
        'citations': citations,
    })

# Display comparison table
header = '| Version | Tokens | Latency | Patterns | Quality | Citations |'
separator = '|---------|--------|---------|----------|---------|-----------|'
rows = []
for r in comparison_rows:
    rows.append(f'| {r["name"]} | {r["tokens"]} | {r["latency"]:.2f}s | {r["patterns"]}/{n_topics*2} | {r["quality"]:.0f}/100 | {r["citations"]} |')

table = '\n'.join([header, separator] + rows)
display(Markdown(table))

# Determine winner
best = max(comparison_rows, key=lambda x: x['quality'])
display(Markdown(f'**Winner:** {best["name"]} (quality score: {best["quality"]:.0f}/100)'))

## Cross-Analysis Version Comparison

**Query:** "What do Mexicans think about their national identity?" | **Patterns requested:** 3 similar + 3 different

| Version | Tokens | Latency | Patterns | Quality | Citations |
|---------|--------|---------|----------|---------|-----------|
| V1 (Baseline) | 2994 | 9.66s | 6/6 | 85/100 | 0 |
| V2 (Optimized) | 2949 | 9.65s | 6/6 | 100/100 | 36 |
| V3 (CoT) | 3103 | 13.88s | 6/6 | 100/100 | 34 |

**Winner:** V2 (Optimized) (quality score: 100/100)

In [38]:
# === TRY YOUR OWN CROSS-ANALYSIS QUERY ===
# Change query and version, then run

YOUR_QUERY = "How connected do Mexicans feel to their communities?"
VERSION = 'v2'  # 'v1', 'v2', or 'v3'
N_PATTERNS = 3

templates = {'v1': PromptTemplates.CROSS_ANALYSIS_V1, 
             'v2': PromptTemplates.CROSS_ANALYSIS_V2, 
             'v3': PromptTemplates.CROSS_ANALYSIS_V3}

prompt = templates[VERSION].format(
    n_topics=N_PATTERNS,
    user_query=YOUR_QUERY,
    results=survey_results_str,
    format_instructions=format_instr
)

display(Markdown(f'### Your Query ({VERSION.upper()}): "{YOUR_QUERY}"'))
content, meta = call_llm(prompt, temperature=0.5)
cleaned = clean_json(content)
result = json.loads(cleaned)

print()
for key, val in result.items():
    if 'PATTERN' in key:
        kind = 'Similar' if 'SIMILAR' in key else 'Different'
        display(Markdown(f'**{kind} Pattern**: {val["TITLE_SUMMARY"]}'))
        display(Markdown(f'  {val["DESCRIPTION"]}'))
        print()

### Your Query (V2): "How connected do Mexicans feel to their communities?"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 2005 prompt + 997 completion = 3002 total
  Latency: 10.56s



**Similar Pattern**: Strong Emotional Connection to Local and Regional Communities

  Respondents show a strong emotional connection to their immediate and regional communities: 37.92% feel 'Mucho' and 42.42% 'Algo' connected to their neighborhood (p3_1|IDE), 39.75% 'Mucho' and 40.08% 'Algo' connected to their locality/town (p3_2|IDE), and 39.58% 'Mucho' and 36.42% 'Algo' connected to their region (p3_4|IDE). These high percentages indicate a consistent pattern of attachment across various local scales.

**Similar Pattern**: High National Pride and Positive National Identity Associations

...

  While 37.92% feel 'Mucho' connected to their neighborhood and 42.42% 'Algo' (p3_1|IDE), only 27.33% feel 'Mucho' and 35.75% 'Algo' connected to the world (p3_6|IDE). Additionally, 22.00% feel 'Poco' and 9.92% 'Nada' connected globally, indicating a weaker sense of connection to the world compared to local communities.

**Different Pattern**: Higher Emotional Attachment to Mexico than to State or Region

  Attachment to Mexico is stronger with 48.17% feeling 'Mucho' and 33.08% 'Algo' connected (p3_5|IDE) compared to state attachment at 41.75% 'Mucho' and 34.50% 'Algo' (p3_3|IDE), and regional connection at 39.58% 'Mucho' and 36.42% 'Algo' (p3_4|IDE). This suggests national identity evokes more intense emotional connection than intermediate territorial identities.

**Different Pattern**: Strong Valuation of Cultural Traditions versus Ambivalent National Identity Associations

  While 73% highly value preserving traditions from their place of origin ('Mucho' in p8|IDE), only 25.25% prominently associate 'México' with 'Orgullo' (pride), and 9.75% answered 'No sabe/ No contesta' regarding national associations (p1_1a_1|IDE). This contrast indicates a stronger connection to cultural traditions than to a clear or uniformly positive national identity.

---
## 5. Expert Summary Prompt

**Purpose:** Given a pattern from cross-analysis, generate expert-level interpretation with domain knowledge.

**Source:** `detailed_analysis.py:225-246` + `meta_prompting.py` (V1, V2)

In [39]:
# === TEST 5: EXPERT SUMMARY ===
# Use a pattern from the V2 cross-analysis above

# Get first similar pattern from V2 results
pattern_key = [k for k in result_v2 if 'SIMILAR_PATTERN_1' in k][0] if any('SIMILAR_PATTERN_1' in k for k in result_v2) else list(result_v2.keys())[0]
pattern = result_v2[pattern_key]

# Build expert implications from the checkpoint data
pattern_var_ids = [v.strip() for v in pattern['VARIABLE_STRING'].split(',')]
implications = []
for vid in pattern_var_ids:
    if vid in checkpoint_data:
        implications.append(checkpoint_data[vid]['implication'])
imp_str = ' * '.join(implications) if implications else 'No expert implications available.'

expert_prompt = f"""
You are a very thorough research assistant working on a survey research project.
The objective is to study public opinion on various topics.
You are fully bilingual in English and Spanish. Reply in English.

Read the EXPERT STATEMENTS from experts about what information they consider important.
Then read the SURVEY SUMMARY and write a one-paragraph reply elaborating on how the results relate to their concerns.
Include as many relevant facts (numbers) as possible, citing QUESTION_IDs in parentheses (e.g., p5_1|IDE).

EXPERT STATEMENTS: * {imp_str}
SURVEY SUMMARY: {pattern['DESCRIPTION']}

Return JSON: {{"EXPERT_REPLY": "your paragraph here"}}
"""

display(Markdown(f'### Expert Summary for Pattern: "{pattern["TITLE_SUMMARY"]}"'))
display(Markdown(f'**Variables:** `{pattern["VARIABLE_STRING"]}`'))
print()

content_expert, meta_expert = call_llm(expert_prompt, temperature=0.7)
cleaned_expert = clean_json(content_expert)
result_expert = json.loads(cleaned_expert)

print()
display(Markdown(f'**Expert Analysis:**'))
display(Markdown(result_expert['EXPERT_REPLY']))
#TODO: explain why assessor gave a score of 16/100
# Quality assessment
quality = assessor.assess_expert_summary_quality(content_expert, result_expert)
display(Markdown(f'*Quality Score: {quality:.0f}/100*'))

### Expert Summary for Pattern: "Strong Emotional Attachment to Mexico and Subnational Identities"

**Variables:** `p3_1|IDE,p3_2|IDE,p3_3|IDE,p3_4|IDE,p3_5|IDE`


  Model: gpt-4.1-mini-2025-04-14
  Tokens: 589 prompt + 218 completion = 807 total
  Latency: 2.43s



**Expert Analysis:**

The survey results closely align with expert concerns about the importance of understanding emotional attachment across various geographic levels to inform community engagement and policy development. The data reveal that a significant portion of respondents express strong emotional connection, with 37.92% feeling 'Mucho' attached to their neighborhood (p3_1|IDE) and 39.75% to their locality (p3_2|IDE). This local attachment is slightly exceeded by feelings toward broader identities, with 41.75% strongly connected to their state (p3_3|IDE) and 39.58% to their region (p3_4|IDE). Notably, national identity elicits the highest attachment, with 48.17% feeling 'Mucho' connected to Mexico as a whole (p3_5|IDE). These findings underscore a consistent pattern of emotional connection that experts in 'identidad y valores' can leverage to tailor cultural policies, strengthen social cohesion, and design initiatives that foster pride and engagement at neighborhood, regional, and national levels.

*Quality Score: 28/100*

---
## 6. Transversal Analysis Prompt

**Purpose:** Synthesize all expert summaries into topic summaries, a general overview, and a direct query answer.

**Source:** `detailed_analysis.py:348-386` + `meta_prompting.py` (V1, V2)

In [40]:
# === TEST 6: TRANSVERSAL ANALYSIS ===
# Simulate having multiple expert summaries from different patterns

# Build mock expert summaries from all patterns found in V2
expert_statements = []
for key, val in result_v2.items():
    if 'PATTERN' in key:
        expert_statements.append(f"{val['TITLE_SUMMARY']}: {val['DESCRIPTION']}")

# Add the real expert reply from test 5
expert_statements.append(result_expert.get('EXPERT_REPLY', ''))

all_statements = ' * '.join(expert_statements)

transversal_prompt = f"""
You are a thorough research assistant and expert in survey research and public opinion.
You are fully bilingual in English and Spanish.

Perform three analyses and return a single JSON dictionary:

1) Read the STATEMENTS from experts. Identify 3 common topics across them.
   Write a one-paragraph summary of each topic, citing numbers and QUESTION_IDs.
   Save as TOPIC_SUMMARIES (dict with topic names as ALL CAPS keys).

2) Write a one-paragraph summary of the most relevant results for a general audience.
   Include facts and QUESTION_IDs. Save as TOPIC_SUMMARY.

3) Write a two-sentence answer to the QUERY summarizing the most important points.
   Do not include numbers, just ideas. Save as QUERY_ANSWER.

STATEMENTS: {all_statements}

QUERY: {query}

Reply in the language of the QUERY. Return only valid JSON.
Return JSON with keys: TOPIC_SUMMARIES, TOPIC_SUMMARY, QUERY_ANSWER
"""

display(Markdown('### Transversal Analysis'))
display(Markdown(f'**Query:** "{query}"'))
display(Markdown(f'**Input:** {len(expert_statements)} expert statements'))
print()

content_trans, meta_trans = call_llm(transversal_prompt, temperature=0.7)
cleaned_trans = clean_json(content_trans)
result_trans = json.loads(cleaned_trans)

print()
display(Markdown('#### Direct Answer'))
display(Markdown(result_trans.get('QUERY_ANSWER', 'N/A')))

display(Markdown('#### General Summary'))
display(Markdown(result_trans.get('TOPIC_SUMMARY', 'N/A')))

display(Markdown('#### Topic Summaries'))
for topic, summary in result_trans.get('TOPIC_SUMMARIES', {}).items():
    display(Markdown(f'**{topic}:** {summary}'))
    print()

### Transversal Analysis

**Query:** "What do Mexicans think about their national identity?"

**Input:** 7 expert statements


  Model: gpt-4.1-mini-2025-04-14
  Tokens: 1075 prompt + 640 completion = 1715 total
  Latency: 5.93s



#### Direct Answer

...

Mexicans express strong emotional attachment to their country at various geographic levels, feeling most connected to Mexico as a whole (48.17%, p3_5|IDE) while also valuing local and regional identities. Pride is the central feeling associated with national identity, with over 63% expressing strong pride (p6|IDE), yet there is also an awareness of negative stereotypes like corruption (p2_1a_1|IDE). Cultural traditions and respect for diversity are highly valued, with over 70% emphasizing the importance of preserving customs and embracing ethnic differences (p8|IDE, p9|IDE). These findings paint a picture of a complex, layered national identity combining pride, cultural preservation, and critical self-awareness.

#### Topic Summaries

**EMOTIONAL ATTACHMENT ACROSS GEOGRAPHIC LEVELS:** Respondents exhibit strong emotional connections at multiple geographic scales, with 37.92% feeling 'Mucho' attached to their neighborhood (p3_1|IDE), 39.75% to their locality (p3_2|IDE), 41.75% to their state (p3_3|IDE), and 39.58% to their region (p3_4|IDE). The highest attachment is to Mexico as a whole, with 48.17% expressing 'Mucho' connection (p3_5|IDE). This pattern reflects deep-rooted attachments that vary from local to national levels, highlighting the layered nature of identity among Mexicans.

**PRIDE AND COMPLEX NATIONAL IDENTITY:** Pride ('Orgullo') emerges as the dominant positive emotion linked to Mexican national identity, mentioned by 25.25% (p1_1a_1|IDE) and seen as the main feeling by 38.33% (p5_1|IDE and p5_1a|IDE). A majority (63.42%) report strong pride in being Mexican (p6|IDE). However, this pride coexists with recognition of negative stereotypes such as corruption cited at 3.58% (p2_1a_1|IDE), indicating a nuanced national identity that embraces both positive self-regard and critical reflection.

**CULTURAL TRADITIONS AND DIVERSITY:** A significant emphasis is placed on valuing cultural traditions and respecting diversity, with 73% rating preservation of traditions as very important (p8|IDE) and 70.42% supporting respect for ethnic and cultural groups (p9|IDE). Additionally, 55.25% agree that a great nation can be built despite cultural differences (p4|IDE). This underscores a shared commitment to cultural heritage and pluralism as foundational elements of Mexican identity.

---
## 7. Meta-Prompting System: A/B Testing & Performance Tracking

**Purpose:** Use the PromptManager to run A/B tests, track performance, and auto-select the best version.

**Source:** `meta_prompting.py` + `prompt_integration.py`

In [41]:
# === TEST 7a: META-PROMPTING PERFORMANCE TRACKING ===

# TODO: explain what prompt_integration is and how it works. What package do they come from?

from meta_prompting import get_prompt_manager, PromptTemplates
from prompt_integration import PromptQualityAssessor, compare_prompt_versions

manager = get_prompt_manager()
assessor = PromptQualityAssessor()

display(Markdown('### Recording performance from tests above...'))

# Record V1 performance
quality_v1 = assessor.assess_cross_analysis_quality(content_v1, result_v1, n_topics * 2)
manager.record_execution('cross_analysis', 'v1', meta_v1['tokens'], meta_v1['latency'], True, quality_v1)
print(f'V1: tokens={meta_v1["tokens"]}, latency={meta_v1["latency"]:.2f}s, quality={quality_v1:.0f}/100')

# Record V2 performance
quality_v2 = assessor.assess_cross_analysis_quality(content_v2, result_v2, n_topics * 2)
manager.record_execution('cross_analysis', 'v2', meta_v2['tokens'], meta_v2['latency'], True, quality_v2)
print(f'V2: tokens={meta_v2["tokens"]}, latency={meta_v2["latency"]:.2f}s, quality={quality_v2:.0f}/100')

# Record V3 performance
quality_v3 = assessor.assess_cross_analysis_quality(content_v3, result_v3, n_topics * 2)
manager.record_execution('cross_analysis', 'v3', meta_v3['tokens'], meta_v3['latency'], True, quality_v3)
print(f'V3: tokens={meta_v3["tokens"]}, latency={meta_v3["latency"]:.2f}s, quality={quality_v3:.0f}/100')

print()
display(Markdown('### Auto-selected best version:'))
best_version = manager._select_best_version('cross_analysis')
display(Markdown(f'**Winner: {best_version}**'))

### Recording performance from tests above...

V1: tokens=2994, latency=9.66s, quality=85/100
V2: tokens=2949, latency=9.65s, quality=100/100
V3: tokens=3103, latency=13.88s, quality=100/100



### Auto-selected best version:

**Winner: v2**

In [42]:
# === TEST 7b: A/B TEST SETUP ===
 # TODO: explain what manager.start_ab_test do? how does it work? what exactly is being compared? 

display(Markdown('### Starting A/B Test: V2 vs V3'))

manager.start_ab_test('cross_analysis', 'v2', 'v3', split_ratio=0.5)
print('A/B test started (50/50 split)')
print()

# Simulate 6 requests and see which version each gets
for i in range(6):
    request_id = f'test_request_{i}'
    version = manager.get_ab_test_version('cross_analysis', request_id)
    print(f'  Request "{request_id}" -> assigned to {version}')

### Starting A/B Test: V2 vs V3

A/B test started (50/50 split)

  Request "test_request_0" -> assigned to v2
  Request "test_request_1" -> assigned to v2
  Request "test_request_2" -> assigned to v2
  Request "test_request_3" -> assigned to v2
  Request "test_request_4" -> assigned to v3
  Request "test_request_5" -> assigned to v2


In [43]:
# === TEST 7c: RUN A/B COMPARISON WITH REAL LLM CALLS ===
ab_query = "How connected do Mexicans feel to their country and local communities?"
n = 2  # Fewer patterns for speed

display(Markdown(f'### A/B Test: "{ab_query}"'))
print()

ab_results = {}
for version_name, template in [('v2', PromptTemplates.CROSS_ANALYSIS_V2), 
                                ('v3', PromptTemplates.CROSS_ANALYSIS_V3)]:
    display(Markdown(f'#### Running {version_name.upper()}...'))
    p = template.format(
        n_topics=n,
        user_query=ab_query,
        results=survey_results_str,
        format_instructions=format_instr
    )
    content, meta = call_llm(p, temperature=0.5)
    cleaned = clean_json(content)
    parsed = json.loads(cleaned)
    # TODO: explain what clean_json() and json.loads() do and why they are necessary
    # TODO: explain what .assess_cross_analysis_quality() and record_execution() -below- do 

    quality = assessor.assess_cross_analysis_quality(content, parsed, n * 2)
    manager.record_execution('cross_analysis', version_name, meta['tokens'], meta['latency'], True, quality)
    
    ab_results[version_name] = {
        'quality': quality,
        'tokens': meta['tokens'],
        'latency': meta['latency'],
        'patterns': len([k for k in parsed if 'PATTERN' in k]),
        'result': parsed
    }
    print()

# Compare
display(Markdown('### A/B Test Results'))
header = '| Metric | V2 (Optimized) | V3 (CoT) |'
sep = '|--------|----------------|----------|'
r2, r3 = ab_results['v2'], ab_results['v3']
rows = [
    f'| Quality Score | {r2["quality"]:.0f}/100 | {r3["quality"]:.0f}/100 |',
    f'| Total Tokens | {r2["tokens"]} | {r3["tokens"]} |',
    f'| Latency | {r2["latency"]:.2f}s | {r3["latency"]:.2f}s |',
    f'| Patterns Found | {r2["patterns"]} | {r3["patterns"]} |',
]
display(Markdown('\n'.join([header, sep] + rows)))

winner = 'v2' if r2['quality'] >= r3['quality'] else 'v3'
display(Markdown(f'**A/B Test Winner: {winner.upper()}**'))

### A/B Test: "How connected do Mexicans feel to their country and local communities?"

#### Running V2...

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 2008 prompt + 770 completion = 2778 total
  Latency: 9.01s



#### Running V3...

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 2024 prompt + 671 completion = 2695 total
  Latency: 7.11s



### A/B Test Results

| Metric | V2 (Optimized) | V3 (CoT) |
|--------|----------------|----------|
| Quality Score | 100/100 | 100/100 |
| Total Tokens | 2778 | 2695 |
| Latency | 9.01s | 7.11s |
| Patterns Found | 4 | 4 |

**A/B Test Winner: V2**

In [44]:
# === TEST 7d: FULL PERFORMANCE REPORT ===

from prompt_integration import print_prompt_report

display(Markdown('### Accumulated Performance Report'))
print_prompt_report()

# Also show auto-selection
print()
display(Markdown(f'**Auto-selected best cross_analysis version: {manager._select_best_version("cross_analysis")}**'))

### Accumulated Performance Report

PROMPT PERFORMANCE REPORT

📊 Overview:
   Total Executions: 10
   Avg Success Rate: 100.0%
   Avg Quality Score: 97.5/100

📝 Cross Analysis:
   Version v1:
      Executions: 2
      Avg Tokens: 2989
      Avg Latency: 10.28s
      Success Rate: 100.0%
      Quality: 92.5/100
   Version v2:
      Executions: 4
      Avg Tokens: 2876
      Avg Latency: 9.45s
      Success Rate: 100.0%
      Quality: 100.0/100
   Version v3:
      Executions: 4
      Avg Tokens: 2945
      Avg Latency: 9.98s
      Success Rate: 100.0%
      Quality: 100.0/100




**Auto-selected best cross_analysis version: v2**

---
## 8. Meta-Prompt Optimizer (LLM-Based Prompt Improvement)

**Purpose:** Use an LLM to analyze and optimize an existing prompt.

**Source:** `meta_prompting.py:282-374`

In [45]:
# === TEST 8: META-PROMPT OPTIMIZATION ===

from meta_prompting import MetaPromptOptimizer

def llm_optimize(prompt):
    content, _ = call_llm(prompt, model='gpt-4.1-mini-2025-04-14', temperature=0.7)
    return content

optimizer = MetaPromptOptimizer(llm_function=llm_optimize)

# Analyze V1 prompt characteristics
display(Markdown('### Analyzing V1 Prompt Characteristics'))
analysis = optimizer.analyze_prompt(PromptTemplates.CROSS_ANALYSIS_V1)
for k, v in analysis.items():
    print(f'  {k}: {v}')

print()
display(Markdown('### Asking LLM to Optimize V1 Prompt'))
optimized, explanation = optimizer.optimize_prompt(
    current_prompt=PromptTemplates.CROSS_ANALYSIS_V1,
    required_fields=['TITLE_SUMMARY', 'VARIABLE_STRING', 'DESCRIPTION'],
    optimization_goals=['Reduce token count by 25%', 'Improve clarity', 'Better JSON compliance']
)

display(Markdown('#### LLM Optimization Suggestions:'))
display(Markdown(optimized[:2000]))

### Analyzing V1 Prompt Characteristics

  token_count_estimate: 184
  instruction_count: 5
  example_count: 0
  has_checklist: False
  has_sections: False
  length: 1192



### Asking LLM to Optimize V1 Prompt

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 432 prompt + 629 completion = 1061 total
  Latency: 8.35s


#### LLM Optimization Suggestions:

Certainly! Below is the optimized prompt followed by a detailed explanation of the changes made to reduce token count, improve clarity, and ensure strict JSON compliance.

---

### Optimized Prompt

You are a research assistant analyzing survey data to answer the QUERY below.

Tasks:  
- Identify top {n_topics} SIMILAR patterns (trends or agreements) and {n_topics} DIFFERENT patterns (contrasts or contradictions) relevant to the QUERY.  
- For each pattern, provide:  
  1. TITLE_SUMMARY: concise, descriptive title (never empty).  
  2. VARIABLE_STRING: comma-separated QUESTION_IDs involved (never empty).  
  3. DESCRIPTION: detailed explanation citing numbers and QUESTION_IDs in parentheses (never empty).

Rules:  
- Never leave fields empty; summarize if data is limited.  
- Use only verified facts; always cite QUESTION_IDs.  
- Do not duplicate patterns across SIMILAR and DIFFERENT lists.  
- Exclude results marked 'NaN' or unavailable.  
- When combining categories (e.g., "like a lot" + "like somewhat"), mention sums only if all data is present.  
- Do not fabricate data; note if patterns are weak.  
- Each pattern must be unique and relate directly to the QUERY.

QUERY: {user_query}  
RESULTS: {results}  

{format_instructions}

Output **strict JSON only**, no markdown or extra text.

---

### Explanation of Changes

1. **Reduced verbosity:**  
   - Changed "Your job is to:" to "Tasks:"  
   - Removed redundant phrases like "a short, descriptive title" → "concise, descriptive title"  
   - Combined related instructions into fewer lines (e.g., rules in a bullet list).

2. **Simplified language for clarity:**  
   - "Use only facts you are sure about" → "Use only verified facts"  
   - "If information is limited, generalize or summarize what is available" → "summarize if data is limited"  
   - "Do NOT repeat the same pattern in multiple fields" → "Do not duplicate patterns across SIMILAR and DIFFERENT lists"

3. **Consistent terminology:**  
   - Consistently use

---
## 9. Full Pipeline Test (End-to-End)

Run the complete prompt chain: Intent -> Grading -> Cross-Analysis -> Expert Summary -> Transversal

In [46]:
# === FULL PIPELINE TEST ===

PIPELINE_QUERY = "What do Mexicans think about cultural diversity and preserving traditions?"

display(Markdown(f'# Full Pipeline: "{PIPELINE_QUERY}"'))
pipeline_start = time.time()

# Step 1: Intent Classification
display(Markdown('## Step 1: Intent Classification'))
intent_result = test_intent_classification(PIPELINE_QUERY)
print()

# Step 2: Variable Grading (grade 5 identity variables)
display(Markdown('## Step 2: Variable Relevance Grading'))
grading_results = {}
test_vars = ['p8|IDE', 'p9|IDE', 'p4|IDE', 'p5_1|IDE', 'p10_1|IDE']
for var_id in test_vars:
    if var_id in checkpoint_data:
        data = checkpoint_data[var_id]
        info = f"SUMMARY: {data['summary']} IMPLICATIONS: {data['implication']}"
        grade_result = test_variable_grading(PIPELINE_QUERY, var_id, info)
        grading_results[var_id] = grade_result
        print()

# Filter relevant variables (grade >= 2)
relevant_vars = {k: v for k, v in grading_results.items() if v['GRADE'] >= 2}
display(Markdown(f'**Relevant variables (grade >= 2):** {list(relevant_vars.keys())}'))
print()

# Full Pipeline: "What do Mexicans think about cultural diversity and preserving traditions?"

## Step 1: Intent Classification

**Input:** "What do Mexicans think about cultural diversity and preserving traditions?"

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 258 prompt + 16 completion = 274 total
  Latency: 0.47s


**Result:** `query_variable_database` (confidence: 1.0)

...


  Model: gpt-4.1-mini-2025-04-14
  Tokens: 366 prompt + 85 completion = 451 total
  Latency: 1.24s


  **p5_1|IDE**: Grade 1/3 🟡 — The survey information provides data on emotional sentiments related to Mexico, highlighting feelings such as pride, concern, and anger. While this touches on general attitudes toward the country, it does not specifically address opinions on cultural diversity or the preservation of traditions. Therefore, it has some connection to the query by reflecting national identity emotions but lacks direct insights into views on cultural diversity and tradition preservation.


  Model: gpt-4.1-mini-2025-04-14
  Tokens: 346 prompt + 83 completion = 429 total
  Latency: 1.54s


  **p10_1|IDE**: Grade 2/3 🟠 — The survey information provides relevant insights into how Mexicans perceive cultural diversity in the workplace, specifically regarding feelings of being diminished due to their customs and culture. While it does not directly address broader opinions on cultural diversity or the preservation of traditions, it offers important data on cultural sensitivity and inclusion, which are key aspects of attitudes toward cultural diversity and tradition preservation.

**Relevant variables (grade >= 2):** ['p8|IDE', 'p9|IDE', 'p4|IDE', 'p10_1|IDE']

In [47]:
# Step 3: Cross-Analysis (using auto-selected best version)
display(Markdown('## Step 3: Cross-Analysis Pattern Extraction'))

best_ver = manager._select_best_version('cross_analysis')
display(Markdown(f'*Using auto-selected version: {best_ver}*'))

pipeline_prompt = manager.get_prompt(
    'cross_analysis',
    version=best_ver,
    variables={
        'n_topics': 2,
        'user_query': PIPELINE_QUERY,
        'results': survey_results_str,
        'format_instructions': format_instr
    }
)

content_pipe, meta_pipe = call_llm(pipeline_prompt, temperature=0.5)
cleaned_pipe = clean_json(content_pipe)
result_pipe = json.loads(cleaned_pipe)

# Record performance
quality_pipe = assessor.assess_cross_analysis_quality(content_pipe, result_pipe, 4)
manager.record_execution('cross_analysis', best_ver, meta_pipe['tokens'], meta_pipe['latency'], True, quality_pipe)

print()
for key, val in result_pipe.items():
    if 'PATTERN' in key:
        kind = 'Similar' if 'SIMILAR' in key else 'Different'
        display(Markdown(f'**{kind}**: {val["TITLE_SUMMARY"]}'))
        display(Markdown(f'> {val["DESCRIPTION"][:300]}'))
        print()

## Step 3: Cross-Analysis Pattern Extraction

*Using auto-selected version: v2*

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 2007 prompt + 704 completion = 2711 total
  Latency: 8.13s



**Similar**: Strong Emotional Attachment to Mexican Identity and Traditions

> A large portion of respondents express strong emotional attachment to Mexico and its cultural traditions: 48.17% feel 'Mucho' connected to Mexico (p3_5|IDE), 63.42% feel strong pride in being Mexican (p6|IDE), and 73% value preserving traditions from their place of origin highly ('Mucho') (p8|IDE). 

...

> Most respondents support cultural diversity and respect for distinct cultures within Mexico: 55.25% agree that a great nation can be built despite having distinct cultures and values (p4|IDE), and 70.42% believe ethnic and cultural groups should have their cultures and customs respected rather than 

**Different**: Higher Attachment to Local and Regional Identity Compared to Global Identity

> Respondents feel stronger emotional connection to their locality (79.83% 'Mucho' or 'Algo' connected, p3_2|IDE), state (76.25% 'Mucho' or 'Algo', p3_3|IDE), and region (76.00% 'Mucho' or 'Algo', p3_4|IDE) than to the world (63.08% 'Mucho' or 'Algo', p3_6|IDE). Additionally, a higher percentage feel 

**Different**: National Pride and Positive Associations Contrast with Notable Uncertainty and Negative Emotions

> While 'Orgullo' (Pride) is the most common association with Mexico at 25.25% (p1_1a_1|IDE) and 38.33% express pride as a feeling towards Mexico (p5_1a|IDE, p5_1|IDE), there is also significant presence of concern (17.00%) and anger (11.25%) emotions (p5_1a|IDE). Additionally, notable uncertainty exi

In [48]:
# Step 4: Expert Summaries for each pattern
display(Markdown('## Step 4: Expert Summaries'))

expert_replies = []
for key, val in result_pipe.items():
    if 'PATTERN' not in key:
        continue
    
    # Get implications for this pattern's variables
    var_ids = [v.strip() for v in val['VARIABLE_STRING'].split(',')]
    imps = [checkpoint_data[v]['implication'] for v in var_ids if v in checkpoint_data]
    imp_text = ' * '.join(imps) if imps else 'No expert implications available.'
    
    ep = f"""
You are a thorough research assistant on a survey project about public opinion.
Bilingual English/Spanish. Reply in English.

Read EXPERT STATEMENTS and SURVEY SUMMARY. Write a one-paragraph reply addressing expert concerns.
Include facts with QUESTION_IDs in parentheses.

EXPERT STATEMENTS: * {imp_text}
SURVEY SUMMARY: {val['DESCRIPTION']}

Return JSON: {{"EXPERT_REPLY": "your paragraph"}}
"""
    
    display(Markdown(f'**Pattern: {val["TITLE_SUMMARY"]}**'))
    ec, em = call_llm(ep, temperature=0.7)
    er = json.loads(clean_json(ec))
    expert_replies.append(er.get('EXPERT_REPLY', ''))
    display(Markdown(f'> {er.get("EXPERT_REPLY", "N/A")[:400]}'))
    print()

## Step 4: Expert Summaries

**Pattern: Strong Emotional Attachment to Mexican Identity and Traditions**

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 384 prompt + 155 completion = 539 total
  Latency: 2.13s


> The survey data strongly supports the expert observations regarding the significance of emotional attachment to national identity in Mexico. Nearly half of respondents (48.17%) report feeling a strong connection to Mexico (p3_5|IDE), while an even larger majority (63.42%) express pride in their Mexican nationality (p6|IDE). Furthermore, 73% highly value preserving traditions from their place of or

**Pattern: Majority Support for Respecting Cultural Diversity Over Assimilation**

...

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 503 prompt + 189 completion = 692 total
  Latency: 2.14s


> The survey data clearly supports the experts' emphasis on the importance of understanding emotional connections at multiple geographic levels for insights into identity and social cohesion. Respondents express the strongest connection to their locality (79.83% 'Mucho' or 'Algo', p3_2|IDE), followed closely by their state (76.25%, p3_3|IDE) and region (76.00%, p3_4|IDE), highlighting the prominence

**Pattern: National Pride and Positive Associations Contrast with Notable Uncertainty and Negative Emotions**

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 443 prompt + 214 completion = 657 total
  Latency: 2.46s


> The survey data underscores the complex emotional landscape surrounding Mexican national identity, which is essential for experts in 'identidad y valores' to consider. While pride remains the dominant association, with 25.25% identifying 'Orgullo' as their primary association (p1_1a_1|IDE) and 38.33% expressing pride as a feeling towards Mexico (p5_1a|IDE, p5_1|IDE), there is a substantial presenc

In [49]:
# Step 5: Transversal Analysis (Final Report)
display(Markdown('## Step 5: Transversal Analysis (Final Report)'))

all_expert = ' * '.join(expert_replies)

final_prompt = f"""
You are an expert research assistant in survey research and public opinion.
Bilingual English/Spanish.

Perform three analyses:

1) TOPIC_SUMMARIES: Identify 3 common topics across STATEMENTS. Write a paragraph each with facts and QUESTION_IDs.
2) TOPIC_SUMMARY: One paragraph for general audience with key facts and QUESTION_IDs.
3) QUERY_ANSWER: Two sentences answering the QUERY (no numbers, just ideas).

STATEMENTS: {all_expert}
QUERY: {PIPELINE_QUERY}

Return JSON: {{"TOPIC_SUMMARIES": {{}}, "TOPIC_SUMMARY": "", "QUERY_ANSWER": ""}}
"""

content_final, meta_final = call_llm(final_prompt, temperature=0.7)
result_final = json.loads(clean_json(content_final))

pipeline_elapsed = time.time() - pipeline_start

print()
display(Markdown('### Query Answer'))
display(Markdown(f'> {result_final.get("QUERY_ANSWER", "N/A")}'))

display(Markdown('### General Summary'))
display(Markdown(result_final.get('TOPIC_SUMMARY', 'N/A')))

display(Markdown('### Topic Summaries'))
for topic, summary in result_final.get('TOPIC_SUMMARIES', {}).items():
    display(Markdown(f'**{topic}:** {summary}'))
    print()

display(Markdown(f'---\n*Full pipeline completed in {pipeline_elapsed:.1f}s*'))

## Step 5: Transversal Analysis (Final Report)

  Model: gpt-4.1-mini-2025-04-14
  Tokens: 815 prompt + 795 completion = 1610 total
  Latency: 9.79s



### Query Answer

> Mexicans highly value cultural diversity and believe in preserving their traditional customs rather than assimilating into a dominant culture. They see multiculturalism as a strength that can contribute to building a great nation.

### General Summary

...

**Emotional Attachment to National Identity:** The survey reveals a strong emotional connection among Mexicans to their national identity. Nearly half of respondents (48.17%) report feeling a strong connection to Mexico (p3_5|IDE), and an even larger majority (63.42%) express pride in their Mexican nationality (p6|IDE). Additionally, 73% highly value preserving traditions from their place of origin (p8|IDE). These findings emphasize the importance of cultural heritage and suggest that fostering national pride and social cohesion through identity-related initiatives can be effective, supporting expert observations on the significance of emotional attachment.

**Cultural Diversity and Multiculturalism:** A majority of Mexicans endorse cultural diversity and the preservation of distinct ethnic and cultural customs. Specifically, 55.25% agree that a great nation can be built despite having diverse cultures and values (p4|IDE), and 70.42% believe that ethnic and cultural groups should preserve their customs rather than assimilate into the dominant culture (p9|IDE). This strong support for multiculturalism aligns with expert calls to promote inclusivity and intercultural dialogue, providing a foundation for policies that foster social unity and respect for cultural diversity.

**Local and Regional Identity Connections:** Survey data highlights strong emotional ties to local and regional identities, with 79.83% feeling a strong connection to their locality (p3_2|IDE), 76.25% to their state (p3_3|IDE), and 76.00% to their region (p3_4|IDE). In contrast, connection to the world is lower at 63.08% (p3_6|IDE), with nearly one-third feeling little or no global connection. These patterns validate expert insights that local and regional attachments are more salient and can be leveraged by policymakers and community leaders to strengthen social cohesion on those geographic levels while addressing weaker global ties.

**Emotional Complexity Surrounding National Identity:** The emotional landscape of Mexican national identity is complex, with pride being dominant yet accompanied by concern, anger, and uncertainty. Pride is identified as the primary association by 25.25% (p1_1a_1|IDE) and as a feeling by 38.33% (p5_1a|IDE, p5_1|IDE). However, 17.00% express concern and 11.25% anger (p5_1a|IDE), with notable ambivalence shown by 9.75% not knowing or not responding in associations (p1_1a_1|IDE) and 6.00% in feelings (p5_1a|IDE). These nuances are critical for designing initiatives that foster pride while addressing underlying emotional complexities.

---
*Full pipeline completed in 33.1s*

In [50]:
# === FINAL PERFORMANCE REPORT ===

display(Markdown('# Final Performance Report'))
print_prompt_report()

display(Markdown(f'\n**Best cross_analysis version (auto-selected): {manager._select_best_version("cross_analysis")}**'))

# Show stored performance data
report = manager.get_performance_report()
display(Markdown(f'\n**Total executions tracked:** {report["summary"]["total_executions"]}'))
display(Markdown(f'**Average success rate:** {report["summary"]["avg_success_rate"]*100:.1f}%'))
display(Markdown(f'**Average quality:** {report["summary"]["avg_quality"]:.1f}/100'))

# Final Performance Report

PROMPT PERFORMANCE REPORT

📊 Overview:
   Total Executions: 11
   Avg Success Rate: 100.0%
   Avg Quality Score: 97.5/100

📝 Cross Analysis:
   Version v1:
      Executions: 2
      Avg Tokens: 2989
      Avg Latency: 10.28s
      Success Rate: 100.0%
      Quality: 92.5/100
   Version v2:
      Executions: 5
      Avg Tokens: 2843
      Avg Latency: 9.18s
      Success Rate: 100.0%
      Quality: 100.0/100
   Version v3:
      Executions: 4
      Avg Tokens: 2945
      Avg Latency: 9.98s
      Success Rate: 100.0%
      Quality: 100.0/100




**Best cross_analysis version (auto-selected): v2**


**Total executions tracked:** 11

**Average success rate:** 100.0%

**Average quality:** 97.5/100

---
## Quick Reference: All Prompts

| # | Prompt | File | Purpose | Test Section |
|---|--------|------|---------|-------------|
| 1 | Intent Classification | `intent_classifier.py:41-76` | Route user queries to handlers | Section 1 |
| 2 | Project Description | `dataset_knowledge.py:140-152` | Answer general project questions | Section 2 |
| 3 | Variable Grading | `variable_selector.py:37-83` | Grade variable relevance (0-3) | Section 3 |
| 4 | Cross-Analysis V1/V2/V3 | `detailed_analysis.py:77-140` + `meta_prompting.py` | Find patterns in data | Section 4 |
| 5 | Expert Summary | `detailed_analysis.py:225-246` | Expert interpretation of patterns | Section 5 |
| 6 | Transversal Analysis | `detailed_analysis.py:348-386` | Cross-topic synthesis | Section 6 |
| 7 | Meta-Optimization | `meta_prompting.py:282-303` | LLM-based prompt improvement | Section 8 |

### Meta-Prompting System

| Component | File | Purpose |
|-----------|------|---------|
| PromptTemplates | `meta_prompting.py:38-211` | Version-controlled prompt templates |
| PromptManager | `meta_prompting.py:381-647` | Version selection, A/B testing, tracking |
| MetaPromptOptimizer | `meta_prompting.py:274-374` | LLM-based prompt optimization |
| PromptQualityAssessor | `prompt_integration.py:21-126` | Quality scoring (0-100) |
| Integration functions | `prompt_integration.py:128-320` | Drop-in replacements with tracking |